In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

import warnings
warnings.filterwarnings('ignore')

In [ ]:

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
# Load the dataset
delivery_path = os.path.join(path, 'Q1_data.csv')#ensure the path is correct
df_delivery = pd.read_csv(delivery_path)#read csv file and conver it to df(table)

In [ ]:
# Task 2: Write your code here:
df_delivery.head()#display first 5 rows

In [ ]:
# Task 3: Write your code here:
df_delivery.info()

In [ ]:
# Task 4: Write your code here:
df_delivery.describe()

In [ ]:
# Task 5: Write your code here:
# delivery_time distribution (target variable)
plt.figure(figsize=(10, 5))
plt.hist(df_delivery['Delivery_Time'].dropna())
plt.title('delivery_time Distribution')
plt.xlabel('delivery_time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:
cols = ['Order_ID', 'Distance_km', 'Weather',	'Traffic_Level',	'Time_of_Day',	'Vehicle_Type',	'Preparation_Time_min',	'Courier_Experience_yrs',	'Delivery_Time']
df_clean = df_delivery[cols].copy()

print(f"Before: {df_clean.shape}")
df_clean = df_clean.drop(columns=['Order_ID'])
print(f"After dropping Order_ID: {df_clean.shape}")

In [ ]:
# Task 2: Write your code here:
# Analyze missing values
missing_percentage = (df_delivery.isnull().sum() / len(df_delivery)) * 100
missing_data = pd.DataFrame({
    'Column': missing_percentage.index,
    'Missing_Percentage': missing_percentage.values
})
missing_data = missing_data[missing_data['Missing_Percentage'] > 0].sort_values('Missing_Percentage', ascending=False)

print("Missing Data Analysis:")
missing_data.head(10)

In [ ]:
####continue task 2
### 5 columns have missing values, Delivery_Time is the highest column missing values with 6% missing, while Weather,Traffic_Level,Time_of_Day, and Courier_Experience_yrs	all are approximely 3% missing

# Missing values
print("Missing values:")
print(df_delivery.isnull().sum())

In [ ]:
#for col in ['Delivery_Time', 'Weather', 'Traffic_Level', 'Time_of_Day', 'Courier_Experience_yrs']:
 #   df_clean['cylinders'] = df_clean['cylinders'].fillna(df_clean['cylinders'].mode()[0])

# Fill cylinders with mode - discrete feature, mode is most representative
#print("Missing values remaining:", df_clean.isnull().sum().sum())

In [ ]:
# Task 3: Write your code here:

def check_duplicates(df):

  duplicates = df.duplicated().sum()

  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates remomved.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_delivery)

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import OneHotEncoder #import OneHotEncoder
print('data before encoding:\n', df_delivery) #show before encoding

onehot_encoder = OneHotEncoder(sparse_output=False) # Instantiate OneHotEncoder
data_onehot_encoded = onehot_encoder.fit_transform(df_delivery) # Apply fit_transform to the copied

print('\nData after encoding:\n', data_onehot_encoded) #show after encoding

In [ ]:
# Task 5: Write your code here:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\nScaled ranges - Min: {X_train_scaled.min():.2f}, Max: {X_train_scaled.max():.2f}")
pd.DataFrame(X_train_scaled, columns=X_train.columns).head(3)

In [ ]:
# Task 6: Write your code here:
import seaborn as sns
def check_target_imbalance(df_delivery, Delivery_Time):
  print("Target Distribution:")
  print(df_delivery[Delivery_Time].value_counts(normalize=True))
  sns.countplot(x=df_delivery[Delivery_Time])
  plt.title("Target Distribution")
  plt.show()

check_target_imbalance(df_delivery, "Delivery_Time")

In [ ]:
# Task 1: Write your code here:
#Define features (X) and target (y)
feature_cols = ['Distance_km', 'Weather',	'Traffic_Level',	'Time_of_Day',	'Vehicle_Type',	'Preparation_Time_min',	'Courier_Experience_yrs']
X = df_clean[feature_cols]
y = df_clean['Delivery_Time']

# Train-test split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

In [ ]:
# Task 2,3,4,5: Write your code here:
kfold = KFold(n_splits=5, shuffle=True, random_state=42)


mae_scores = []

for train_idx, val_idx in kfold.split(X_train_scaled):
    X_fold_train, X_fold_val = X_train_scaled[train_idx], X_train_scaled[val_idx]
    y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]


model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
model.fit(X_train_scaled, y_train)
print("Model trained!")

    # Train and predict
    #model.fit(X_fold_train, y_fold_train)
y_fold_pred = model.predict(X_fold_val)

    # Calculate metrics
mae_scores.append(mean_absolute_error(y_fold_val, y_fold_pred))

mae_scores = np.array(mae_scores)

print(f"5-Fold CV Results:")
print(f"MAE:  ${mae_scores.mean():,.2f}")

In [ ]:
# Task 1: Write your code here:
# Feature importance
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:


In [ ]:
# Task Bonus: Write your code here: